In [9]:
import torch
from torch import optim, nn, utils, Tensor
import lightning as L
from lightning.pytorch.callbacks.early_stopping import EarlyStopping
import torchvision
print(f"Torch version: {torch.__version__}")
print(f"Torchvision version: {torchvision.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

Torch version: 2.10.0+cu126
Torchvision version: 0.25.0+cu126
CUDA available: True


In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


# Data Loading

In [11]:
from torchvision.transforms import v2
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, random_split, Subset
import torch
import lightning as L

# 1. Transformaciones BASE (CPU)
base_transforms = v2.Compose([
    v2.Grayscale(num_output_channels=1),
    v2.Resize((224, 224)),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True)
])

# 2. Transformaciones de Aumento (GPU)
gpu_train_transforms = v2.Compose([
    v2.RandomHorizontalFlip(p=0.5),
    v2.RandomVerticalFlip(p=0.2),
    v2.RandomRotation(20),
    v2.Normalize(mean=[0.5], std=[0.5])
])

gpu_val_test_transforms = v2.Compose([
    v2.Normalize(mean=[0.5], std=[0.5])
])

# 3. Definición del DataModule (Recomendado por Lightning)
class AnukaDataModule(L.LightningDataModule):
    def __init__(self, dataset_path, batch_size=32, num_workers=4):
        super().__init__()
        self.dataset_path = dataset_path
        self.batch_size = batch_size
        self.num_workers = num_workers
        self.base_transforms = base_transforms

    def setup(self, stage=None):
        dataset = ImageFolder(root=self.dataset_path, transform=self.base_transforms)
        total_size = len(dataset)
        indices = list(range(total_size))
        train_indices, val_indices, test_indices = random_split(
            indices, [0.8, 0.1, 0.1], generator=torch.Generator().manual_seed(42)
        )
        self.train_ds = Subset(dataset, train_indices)
        self.val_ds = Subset(dataset, val_indices)
        self.test_ds = Subset(dataset, test_indices)
        self.classes = dataset.classes

    def train_dataloader(self):
        return DataLoader(self.train_ds, batch_size=self.batch_size, shuffle=True, 
                          num_workers=self.num_workers, pin_memory=True, persistent_workers=True)

    def val_dataloader(self):
        return DataLoader(self.val_ds, batch_size=self.batch_size, shuffle=False, 
                          num_workers=self.num_workers, pin_memory=True, persistent_workers=True)

    def test_dataloader(self):
        return DataLoader(self.test_ds, batch_size=self.batch_size, shuffle=False, 
                          num_workers=self.num_workers, pin_memory=True, persistent_workers=True)

# Instanciamos el DataModule
dm = AnukaDataModule(dataset_path='dataset/anuka1200', batch_size=32, num_workers=4)
dm.setup()
print(f"Classes found: {dm.classes}")


Classes found: ['Tipo A: Kunzea', 'Tipo B: Lepto']


# Visualization

In [12]:
import matplotlib.pyplot as plt
import random
import numpy as np

# 1. Select a random index
idx = random.randint(0, len(dataset) - 1)
image, label = dataset[idx]

# 2. Prepare the image for visualization (Ya está en [0, 1] desde ToDtype)
image_vis = image.numpy().squeeze() # (1, 224, 224) -> (224, 224)

# 3. Display
plt.figure(figsize=(6, 6))
plt.imshow(image_vis, cmap='gray')
plt.title(f"Clase: {dataset.classes[label]} (index {idx}) Shape: {image.shape}")
plt.axis('off')
plt.show()

NameError: name 'dataset' is not defined

# Entrenamiento

## Definición del modelo.
Usaremos Pytorch Lightning, un wrapper de PyTorch que nos facilita el entrenamiento y la organización del código. Para ello, definiremos una clase que herede de `pl.LightningModule` y que implemente los métodos necesarios para el entrenamiento, validación y test.

```python

In [13]:
class RedNeuronal(L.LightningModule):
    def __init__(self, model, is_cnn=False, lr=1e-3, train_transforms=None, val_transforms=None):
        super().__init__()
        self.save_hyperparameters(ignore=['model', 'train_transforms', 'val_transforms'])
        self.model = model
        self.loss_fn = nn.BCEWithLogitsLoss()
        self.is_cnn = is_cnn
        self.lr = lr
        self.train_transforms = train_transforms
        self.val_transforms = val_transforms

    def on_after_batch_transfer(self, batch, dataloader_idx):
        # Este hook mueve las transformaciones costosas a la GPU (CUDA)
        x, y = batch
        if self.trainer.training and self.train_transforms:
            x = self.train_transforms(x)
        elif self.val_transforms:
            x = self.val_transforms(x)
        return x, y

    def training_step(self, batch, batch_idx):
        x, y = batch
        if not self.is_cnn:
            x = x.view(x.size(0), -1)  
        logits = self.model(x).squeeze(dim=1)
        y = y.float()
        loss = self.loss_fn(logits, y)
        preds = (torch.sigmoid(logits) > 0.5).float()
        accuracy = (preds == y).float().mean()
        self.log('train_loss', loss, prog_bar=True)
        self.log('train_accuracy', accuracy, prog_bar=True)
        return loss
    
    def test_step(self, batch, batch_idx):
        x, y = batch
        if not self.is_cnn:
            x = x.view(x.size(0), -1)  
        logits = self.model(x).squeeze(dim=1)
        y = y.float()
        loss = self.loss_fn(logits, y)
        preds = (torch.sigmoid(logits) > 0.5).float()
        accuracy = (preds == y).float().mean()
        self.log('test_loss', loss, prog_bar=True)
        self.log('test_accuracy', accuracy, prog_bar=True)

    def validation_step(self, batch, batch_idx):    
        x, y = batch
        if not self.is_cnn:
            x = x.view(x.size(0), -1)  
        logits = self.model(x).squeeze(dim=1)
        y = y.float()
        loss = self.loss_fn(logits, y)
        preds = (torch.sigmoid(logits) > 0.5).float()
        accuracy = (preds == y).float().mean()
        self.log('val_accuracy', accuracy, prog_bar=True)
        self.log('val_loss', loss, prog_bar=True)

    def configure_optimizers(self):
        optimizer = optim.Adam(self.parameters(), lr=self.lr)
        return optimizer

In [14]:
from lightning.pytorch.tuner import Tuner

# 1. Modelo de estrés
modelo_stress = nn.Sequential(
    nn.Conv2d(1, 16, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Conv2d(16, 32, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Flatten(),
    nn.Linear(32 * 56 * 56, 16),
    nn.ReLU(),
    nn.Linear(16, 1)
)

modelo_lig_stress = RedNeuronal(modelo_stress, is_cnn=True, 
                               train_transforms=gpu_train_transforms, 
                               val_transforms=gpu_val_test_transforms)

# 2. Tuner con el DataModule
trainer_finder = L.Trainer(accelerator="gpu", devices=1)
tuner = Tuner(trainer_finder)

print("Buscando el Batch Size máximo...")
try:
    # scale_batch_size ahora usa el datamodule directamente
    new_batch_size = tuner.scale_batch_size(modelo_lig_stress, 
                                            datamodule=dm,
                                            mode="power",
                                            method="fit",
                                            init_val=32, 
                                            max_trials=8)
    
    print(f"\n¡Batch Size óptimo encontrado!: {new_batch_size}")
    dm.batch_size = new_batch_size
    print(f"DataModule actualizado con batch_size={dm.batch_size}")
except Exception as e:
    print(f"Error: {e}")
    print("Manteniendo 32.")


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
`weights_only` was not set, defaulting to `False`.


Buscando el Batch Size máximo...


`Trainer.fit` stopped: `max_steps=3` reached.
Batch size 32 succeeded, trying batch size 64
`Trainer.fit` stopped: `max_steps=3` reached.
Batch size 64 succeeded, trying batch size 128
`Trainer.fit` stopped: `max_steps=3` reached.
Batch size 128 succeeded, trying batch size 256
`Trainer.fit` stopped: `max_steps=3` reached.
Batch size 256 succeeded, trying batch size 512
`Trainer.fit` stopped: `max_steps=3` reached.
Batch size 512 succeeded, trying batch size 1024
Batch size 1024 failed, trying batch size 512
Finished batch size finder, will continue with full run using batch size 512
Restoring states from the checkpoint path at /home/diego/UNED/MineriaDeDatosUNED/PEC3/.scale_batch_size_d7cd6221-1ed6-4bb4-8f48-5f7a6a147a26.ckpt
Restored all states from the checkpoint at /home/diego/UNED/MineriaDeDatosUNED/PEC3/.scale_batch_size_d7cd6221-1ed6-4bb4-8f48-5f7a6a147a26.ckpt



¡Batch Size óptimo encontrado!: 512
DataModule actualizado con batch_size=512


In [15]:
from lightning.pytorch.loggers import CSVLogger
from lightning.pytorch.callbacks.early_stopping import EarlyStopping
max_epochs = 100

## Modelo 1: NET-1 (Regresión Logística / Red Neuronal Unicapa)

In [16]:
modelo_net1 = nn.Sequential(nn.Linear(224 * 224, 1))
modelo_net1 = RedNeuronal(modelo_net1, is_cnn=False, train_transforms=gpu_train_transforms, val_transforms=gpu_val_test_transforms)

logger_net1 = CSVLogger("logs", name="net1")
trainer_net1 = L.Trainer(max_epochs=max_epochs, logger=logger_net1, callbacks=[EarlyStopping(monitor="val_loss", mode="min", patience=5)])
from lightning.pytorch.tuner import Tuner
tuner = Tuner(trainer_net1)
lr_finder = tuner.lr_find(modelo_net1, datamodule=dm)
modelo_net1.lr = lr_finder.suggestion()
print(f"Mejor LR sugerida para NET-1: {modelo_net1.lr}")
trainer_net1.fit(model=modelo_net1, datamodule=dm)
trainer_net1.test(model=modelo_net1, datamodule=dm)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
`weights_only` was not set, defaulting to `False`.


Finding best initial lr:   0%|          | 0/100 [00:00<?, ?it/s]

LR finder stopped early after 77 steps due to diverging loss.
Restoring states from the checkpoint path at /home/diego/UNED/MineriaDeDatosUNED/PEC3/.lr_find_fa551b2c-6720-42f0-aea4-6af50730b265.ckpt
Restored all states from the checkpoint at /home/diego/UNED/MineriaDeDatosUNED/PEC3/.lr_find_fa551b2c-6720-42f0-aea4-6af50730b265.ckpt
Learning rate set to 1.445439770745928e-06
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Mejor LR sugerida para NET-1: 1.445439770745928e-06


┏━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name             ┃ Type              ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model            │ Sequential        │ 50.2 K │ train │     0 │
│ 1 │ loss_fn          │ BCEWithLogitsLoss │      0 │ train │     0 │
│ 2 │ train_transforms │ Compose           │      0 │ train │     0 │
│ 3 │ val_transforms   │ Compose           │      0 │ train │     0 │
└───┴──────────────────┴───────────────────┴────────┴───────┴───────┘

...

Output()

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│       test_accuracy       │    0.5250000357627869     │
│         test_loss         │    0.6944050192832947     │
└───────────────────────────┴───────────────────────────┘

[{'test_loss': 0.6944050192832947, 'test_accuracy': 0.5250000357627869}]

## Modelo 2: NET-2 (Perceptrón Multicapa - MLP)

In [17]:
modelo_net2 = nn.Sequential(nn.Linear(224 * 224, 128), nn.ReLU(), nn.Linear(128, 64), nn.ReLU(), nn.Linear(64, 1))
modelo_net2 = RedNeuronal(modelo_net2, is_cnn=False, train_transforms=gpu_train_transforms, val_transforms=gpu_val_test_transforms)

logger_net2 = CSVLogger("logs", name="net2")
trainer_net2 = L.Trainer(max_epochs=max_epochs, logger=logger_net2, callbacks=[EarlyStopping(monitor="val_loss", mode="min", patience=5)])
from lightning.pytorch.tuner import Tuner
tuner = Tuner(trainer_net2)
lr_finder = tuner.lr_find(modelo_net2, datamodule=dm)
modelo_net2.lr = lr_finder.suggestion()
print(f"Mejor LR sugerida para NET-2: {modelo_net2.lr}")
trainer_net2.fit(model=modelo_net2, datamodule=dm)
trainer_net2.test(model=modelo_net2, datamodule=dm)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
`weights_only` was not set, defaulting to `False`.


Finding best initial lr:   0%|          | 0/100 [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_steps=100` reached.
Restoring states from the checkpoint path at /home/diego/UNED/MineriaDeDatosUNED/PEC3/.lr_find_dbbdf599-1807-44d2-9717-5ceaceb35784.ckpt
Restored all states from the checkpoint at /home/diego/UNED/MineriaDeDatosUNED/PEC3/.lr_find_dbbdf599-1807-44d2-9717-5ceaceb35784.ckpt
Learning rate set to 3.9810717055349735e-05
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Mejor LR sugerida para NET-2: 3.9810717055349735e-05


┏━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name             ┃ Type              ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model            │ Sequential        │  6.4 M │ train │     0 │
│ 1 │ loss_fn          │ BCEWithLogitsLoss │      0 │ train │     0 │
│ 2 │ train_transforms │ Compose           │      0 │ train │     0 │
│ 3 │ val_transforms   │ Compose           │      0 │ train │     0 │
└───┴──────────────────┴───────────────────┴────────┴───────┴───────┘

...

Output()

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│       test_accuracy       │    0.8083333969116211     │
│         test_loss         │     0.457125186920166     │
└───────────────────────────┴───────────────────────────┘

[{'test_loss': 0.457125186920166, 'test_accuracy': 0.8083333969116211}]

## Modelo 3: CNN Simple (Base)

In [18]:
modelo_cnn = nn.Sequential(
    nn.Conv2d(1, 16, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Conv2d(16, 32, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Flatten(),
    nn.Linear(32 * 56 * 56, 16),
    nn.ReLU(),
    nn.Linear(16, 1)
)

modelo_cnn = RedNeuronal(modelo_cnn, is_cnn=True, train_transforms=gpu_train_transforms, val_transforms=gpu_val_test_transforms)

logger_cnn = CSVLogger("logs", name="cnn_base")
trainer_cnn = L.Trainer(max_epochs=max_epochs, logger=logger_cnn, callbacks=[EarlyStopping(monitor="val_loss", mode="min", patience=5)])
from lightning.pytorch.tuner import Tuner
tuner = Tuner(trainer_cnn)
lr_finder = tuner.lr_find(modelo_cnn, datamodule=dm)
modelo_cnn.lr = lr_finder.suggestion()
print(f"Mejor LR sugerida para CNN Base: {modelo_cnn.lr}")
trainer_cnn.fit(model=modelo_cnn, datamodule=dm)
trainer_cnn.test(model=modelo_cnn, datamodule=dm)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
`weights_only` was not set, defaulting to `False`.


Finding best initial lr:   0%|          | 0/100 [00:00<?, ?it/s]

Restoring states from the checkpoint path at /home/diego/UNED/MineriaDeDatosUNED/PEC3/.lr_find_49b53fc4-c956-4412-9250-789561dce157.ckpt
/home/diego/.pyenv/versions/3.13.3/envs/MineriaDeDatos/lib/python3.13/site-packages/lightning/pytorch/trainer/call.py:283: Be aware that when using `ckpt_path`, callbacks used to create the checkpoint need to be provided during `Trainer` instantiation. Please add the following callbacks: ["EarlyStopping{'monitor': 'val_loss', 'mode': 'min'}", "ModelCheckpoint{'monitor': None, 'mode': 'min', 'every_n_train_steps': 0, 'every_n_epochs': 1, 'train_time_interval': None}"].
Restored all states from the checkpoint at /home/diego/UNED/MineriaDeDatosUNED/PEC3/.lr_find_49b53fc4-c956-4412-9250-789561dce157.ckpt


OutOfMemoryError: CUDA out of memory. Tried to allocate 1.53 GiB. GPU 0 has a total capacity of 7.91 GiB of which 1.28 GiB is free. Process 8938 has 162.00 MiB memory in use. Including non-PyTorch memory, this process has 5.88 GiB memory in use. Of the allocated memory 3.01 GiB is allocated by PyTorch, and 2.74 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

## Modelo 4: CNN Variante A (BatchNorm + Dropout)

In [ ]:
modelo_cnn_a = nn.Sequential(
    nn.Conv2d(1, 16, kernel_size=3, padding=1),
    nn.BatchNorm2d(16),
    nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Conv2d(16, 32, kernel_size=3, padding=1),
    nn.BatchNorm2d(32),
    nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Flatten(),
    nn.Linear(32 * 56 * 56, 16),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(16, 1)
)

modelo_cnn_a = RedNeuronal(modelo_cnn_a, is_cnn=True, train_transforms=gpu_train_transforms, val_transforms=gpu_val_test_transforms)

logger_cnn_a = CSVLogger("logs", name="cnn_var_a")
trainer_cnn_a = L.Trainer(max_epochs=max_epochs, logger=logger_cnn_a, callbacks=[EarlyStopping(monitor="val_loss", mode="min", patience=5)])
from lightning.pytorch.tuner import Tuner
tuner = Tuner(trainer_cnn_a)
lr_finder = tuner.lr_find(modelo_cnn_a, datamodule=dm)
modelo_cnn_a.lr = lr_finder.suggestion()
print(f"Mejor LR sugerida para CNN Variante A: {modelo_cnn_a.lr}")
trainer_cnn_a.fit(model=modelo_cnn_a, datamodule=dm)
trainer_cnn_a.test(model=modelo_cnn_a, datamodule=dm)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
`weights_only` was not set, defaulting to `False`.


Finding best initial lr:   0%|          | 0/100 [00:00<?, ?it/s]

Restoring states from the checkpoint path at /home/diego/UNED/MineriaDeDatosUNED/PEC3/.lr_find_7d129673-4d64-419a-ae46-9d82966b7a72.ckpt
Restored all states from the checkpoint at /home/diego/UNED/MineriaDeDatosUNED/PEC3/.lr_find_7d129673-4d64-419a-ae46-9d82966b7a72.ckpt


OutOfMemoryError: CUDA out of memory. Tried to allocate 1.53 GiB. GPU 0 has a total capacity of 7.91 GiB of which 1.52 GiB is free. Process 8938 has 162.00 MiB memory in use. Including non-PyTorch memory, this process has 5.60 GiB memory in use. Of the allocated memory 4.52 GiB is allocated by PyTorch, and 955.44 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

## Modelo 5: CNN Variante B (Más profunda con canales 64 y Dropout)

In [ ]:
modelo_cnn_b = nn.Sequential(
    nn.Conv2d(1, 16, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Conv2d(16, 32, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Conv2d(32, 64, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Flatten(),
    nn.Linear(64 * 28 * 28, 64),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(64, 1)
)

modelo_cnn_b = RedNeuronal(modelo_cnn_b, is_cnn=True, train_transforms=gpu_train_transforms, val_transforms=gpu_val_test_transforms)

logger_cnn_b = CSVLogger("logs", name="cnn_var_b")
trainer_cnn_b = L.Trainer(max_epochs=max_epochs, logger=logger_cnn_b, callbacks=[EarlyStopping(monitor="val_loss", mode="min", patience=5)])
from lightning.pytorch.tuner import Tuner
tuner = Tuner(trainer_cnn_b)
lr_finder = tuner.lr_find(modelo_cnn_b, datamodule=dm)
modelo_cnn_b.lr = lr_finder.suggestion()
print(f"Mejor LR sugerida para CNN Variante B: {modelo_cnn_b.lr}")
trainer_cnn_b.fit(model=modelo_cnn_b, datamodule=dm)
trainer_cnn_b.test(model=modelo_cnn_b, datamodule=dm)